# Stage 3 — Preference Tuning with DPO

**Starts from:** the instruction-tuned model produced by `02_instruction_finetuning_lora.ipynb`
**Data:** [`qiaojin/PubMedQA`](https://huggingface.co/datasets/qiaojin/PubMedQA) `pqa_labeled` —
the same 800 training records, plus answers the model writes for itself
**Hardware:** free-tier Colab T4 · **Runtime:** ~35 minutes

---

## What this notebook does

Stages 1 and 2 both asked the same question of every token: **what comes next?** Stage 1 asked it
of raw abstract prose, stage 2 asked it only of the response span. Preference tuning asks something
different:

> Given two complete answers to the same prompt, **which one is better?**

There is no gold target any more, only a ranking. Direct Preference Optimization turns that ranking
into a loss you can backpropagate, with no reward model and no reinforcement learning:

$$\mathcal{L}_\text{DPO} = -\log \sigma\!\left(\beta\left[\log\frac{\pi_\theta(y^+\!\mid x)}{\pi_\text{ref}(y^+\!\mid x)} - \log\frac{\pi_\theta(y^-\!\mid x)}{\pi_\text{ref}(y^-\!\mid x)}\right]\right)$$

$\pi_\theta$ is the model being trained, $\pi_\text{ref}$ is the frozen stage-2 model it started
from, and $\beta$ controls how far it is allowed to drift from it.

```
base ──► notebook 01 ──► + domain ──► notebook 02 ──► + instruct ──► notebook 03 ──► + preference
         raw abstracts                (instruction, response)        (chosen, rejected)
         loss on every token          loss on the response           loss on the *margin*
```

## Where the pairs come from

Most DPO walkthroughs hand-write a dozen preference pairs, which teaches the API and nothing else.
This notebook **mines its pairs from the stage-2 model's own mistakes**: sample two answers per
training question, grade both against the expert label, and pair a correct sample against an
incorrect one. The negatives are then drawn from the distribution actually being optimised, which
is what the DPO derivation assumes and what hand-written negatives are not.

## What to expect

This is the first stage in the repo where **"no measurable change" is a likely and legitimate
result**. A few hundred pairs on a 1B model is very little. The notebook is built to detect that
honestly: it scores the same 200 held-out articles notebook 02 used and reports a **paired
significance test**, not two accuracy numbers printed next to each other.

**Set your runtime to a GPU first:** Runtime → Change runtime type → T4 GPU.

## 1. Runtime and dependencies

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "no GPU found")

In [ ]:
# Pinned to match requirements-colab.txt. torch is deliberately left alone: Colab ships a
# CUDA-matched build and replacing it is slow and fragile.
#
# trl is new in this notebook, and it is the only place in the repo that uses it. Stages 1 and 2
# hand-write their training loops on plain transformers.Trainer so the masking and packing stay
# visible on the page. The DPO loss is a different proposition — subtle, easy to get quietly
# wrong, and TRL's is the reference implementation. Everything around the loss (the pairs, the
# reference policy, every metric) is still built and checked by hand below.
%pip install -q "transformers==5.15.0" "peft==0.20.0" "datasets==5.0.1" "accelerate==1.14.0" "trl==0.29.1"
# Colab preinstalls torchao 0.10.x. peft's optional torchao integration RAISES rather than
# degrading gracefully when it finds a version below its 0.16.0 minimum, and the check fires
# deep inside get_peft_model(). Nothing here uses torchao, so remove it.
%pip uninstall -q -y torchao
print("\nRestart the runtime if Colab asks you to, then run this cell again and continue.")

In [ ]:
import gc, json, math, random, re
from pathlib import Path
from collections import Counter

import torch
import transformers
import peft
import trl
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), "No CUDA device. Runtime > Change runtime type > T4 GPU."

# bfloat16 needs Ampere (sm_80) or later; a T4 is Turing (sm_75). Checking compute capability
# directly, because torch.cuda.is_bf16_supported() has returned True on Turing.
SUPPORTS_BF16 = torch.cuda.get_device_capability()[0] >= 8
DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
TF_MAJOR = int(transformers.__version__.split(".")[0])
DTYPE_KW = "dtype" if TF_MAJOR >= 5 else "torch_dtype"

MODEL_ID = "unsloth/Llama-3.2-1B"

print(f"transformers {transformers.__version__} | peft {peft.__version__} | trl {trl.__version__}")
print(f"GPU {torch.cuda.get_device_name(0)} | compute dtype {DTYPE} | from_pretrained keyword {DTYPE_KW!r}")

# Fail fast on the torchao clash, rather than eight cells from now inside get_peft_model().
import importlib.metadata as _md


def _ver(s):
    return tuple(int(p) for p in s.split("+")[0].split(".")[:3] if p.isdigit())


try:
    _ta = _md.version("torchao")
    if _ver(_ta) < (0, 16, 0):
        raise RuntimeError(
            f"torchao {_ta} is too old for peft {peft.__version__} (needs >= 0.16.0).\n"
            "Nothing in this notebook uses torchao. Fix it with:\n"
            "    !pip uninstall -y torchao\n"
            "then Runtime > Restart session, then Runtime > Run all."
        )
    print(f"torchao      {_ta} (compatible)")
except _md.PackageNotFoundError:
    print("torchao      absent - fine, nothing here uses it")

## 2. Find the stage-1 and stage-2 adapters

This notebook needs **both** earlier adapters, and the reason is worth stating plainly: notebook 02
saved only its own LoRA, and that LoRA was trained on top of the *stage-1-merged* weights. Attaching
it to the pristine base would apply it to weights it has never seen.

So the stage-2 policy has to be rebuilt exactly the way it was created — base, merge stage 1, merge
stage 2 — and we probe for `adapter_config.json` rather than for the folder, because an empty
directory left behind by a half-finished Drive sync looks identical from the outside.

Unlike notebook 02, there is no `SKIP` switch here. Stage 3 tunes the model stage 2 produced; there
is no meaningful way to skip the thing being tuned.

In [ ]:
# Colab wipes /content on disconnect, so notebooks 01 and 02 copied their adapters to Drive.
if not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        pass


def find_adapter(name):
    """Prefer the Drive copy, fall back to this session's local outputs."""
    for candidate in (Path(f"/content/drive/MyDrive/finetuning-demo/{name}"),
                      Path(f"/content/outputs/{name}")):
        if (candidate / "adapter_config.json").exists():
            return candidate
    return None


STAGE1_DIR = find_adapter("stage1-domain-lora")
STAGE2_DIR = find_adapter("stage2-instruct-lora")

for label, path in [("stage 1 (domain)", STAGE1_DIR), ("stage 2 (instruct)", STAGE2_DIR)]:
    print(f"{label:20s} {path if path else 'NOT FOUND'}")

assert STAGE1_DIR and STAGE2_DIR, (
    "Both adapters are required. Run notebooks 01 and 02 first — this stage tunes the model they "
    "produced, and there is nothing to preference-tune without it."
)

cfg = json.loads((STAGE2_DIR / "adapter_config.json").read_text())
print("\nstage-2 adapter:")
for key in ["base_model_name_or_path", "r", "lora_alpha", "lora_dropout", "target_modules"]:
    print(f"  {key:26s} {cfg.get(key)}")

## 3. Rebuild the stage-2 policy

Two merges, in order: base → `+ domain` → `+ instruct`. What comes out is a plain
`LlamaForCausalLM` with both stages folded into the weights and no PEFT wrapper left over — the
same verified-merge pattern notebook 02 used, run twice.

That merged model then plays **two roles at once** for the rest of the notebook:

1. It is the **initial policy** — what a fresh LoRA gets attached to and trained from.
2. It is the frozen anchor behind the **reference policy** $\pi_\text{ref}$ — what the model is
   penalised for drifting away from.

Ordinarily role 2 costs a second full copy of the model in GPU memory. Because the base weights are
frozen and shared, it costs a copy of the *adapter* instead — about 45 MB rather than 2.5 GB.
Section 9 shows exactly how TRL arranges that, and asserts it rather than assuming it.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
assert tokenizer.pad_token_id is not None, "No pad token — you would need to alias EOS here"
assert tokenizer.pad_token_id != tokenizer.eos_token_id, "PAD and EOS must stay distinct"

# LEFT padding here, unlike notebooks 01 and 02 — and the honest reason is generation, not TRL.
#
# Those notebooks right-pad because they train on (prompt, target) sequences scored per position,
# where trailing pad is masked out of the loss. This notebook never does that. What it does do is
# generate in batches, twice: once to mine preference pairs and once to score the held-out set.
# Batched generation needs every sequence to END at the same column, because that column is where
# the model continues from. Right padding there produces fluent nonsense continuing from <pad>.
#
# TRL is indifferent: its preference collator passes padding_side explicitly on every call (left
# for prompts, right for completions), so it neither needs nor reads this setting.
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **{DTYPE_KW: DTYPE}).to("cuda")
model.config.pad_token_id = tokenizer.pad_token_id

# Snapshot a weight both adapters target, so each merge can be proven rather than trusted.
PROBE_WEIGHT = "model.layers.8.mlp.down_proj.weight"
w_base = dict(model.named_parameters())[PROBE_WEIGHT].detach().clone()

print(f"pad token {tokenizer.pad_token!r} (id {tokenizer.pad_token_id}) | "
      f"padding side {tokenizer.padding_side}")

In [ ]:
# Merge 1 of 2 — domain adaptation, onto the pristine base.
model = PeftModel.from_pretrained(model, STAGE1_DIR)
model = model.merge_and_unload()
w_stage1 = dict(model.named_parameters())[PROBE_WEIGHT].detach().clone()

# Merge 2 of 2 — instruction tuning, onto the merged stage-1 weights.
model = PeftModel.from_pretrained(model, STAGE2_DIR)
model = model.merge_and_unload()
w_stage2 = dict(model.named_parameters())[PROBE_WEIGHT].detach().clone()

# Both merges must have moved the probe weight. A silent no-op here is the classic failure mode:
# AutoModelForCausalLM.from_pretrained(adapter_dir) returns the base model and quietly ignores the
# adapter files sitting beside it, and then every number downstream describes the wrong model.
assert not torch.equal(w_base, w_stage1), "stage-1 merge changed nothing — the adapter was not applied"
assert not torch.equal(w_stage1, w_stage2), "stage-2 merge changed nothing — the adapter was not applied"

print(f"probe tensor {PROBE_WEIGHT}")
print(f"  base    -> +domain    max |delta| {(w_stage1.float() - w_base.float()).abs().max():.3e}")
print(f"  +domain -> +instruct  max |delta| {(w_stage2.float() - w_stage1.float()).abs().max():.3e}")
print(f"\n{type(model).__name__} — no PEFT wrapper, both stages folded into the weights.")

del w_base, w_stage1, w_stage2
gc.collect(); torch.cuda.empty_cache()
print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 4. Rebuild the exact stage-2 split

Same dataset, same constants, same seed, same stratified split as notebook 02 — so the 800 training
articles and the 200 held-out articles come back as *the same articles*. That buys two things:

- pairs are mined only from the **training 800**, so the held-out 200 stay unseen through a third
  stage of training
- the final accuracy number is directly comparable to notebook 02's, because it is literally the
  same test

`scripts/validate_data.py` parses this notebook and fails if any of these constants drift from
`scripts/prepare_data.py`.

In [ ]:
from datasets import load_dataset

# --- constants mirrored in scripts/prepare_data.py and notebooks 01 and 02 ---
DATASET = "qiaojin/PubMedQA"
SEED = 20260815
EVAL_FRACTION = 0.20      # -> 800 train / 200 eval

TASK = (
    "Answer the research question using only the abstract provided. "
    'Begin your reply with "Answer:" followed by yes, no, or maybe, '
    "then justify it in one or two sentences."
)

labeled = load_dataset(DATASET, "pqa_labeled", split="train")


def join_sections(context) -> str:
    """Render an abstract's sections as `LABEL: text` blocks."""
    labels = context.get("labels") or []
    parts = []
    for i, text in enumerate(context["contexts"]):
        text = " ".join(text.split())
        if not text:
            continue
        label = (labels[i] if i < len(labels) else "").strip()
        parts.append(f"{label}: {text}" if label else text)
    return "\n\n".join(parts)


records = []
for row in labeled:
    abstract = join_sections(row["context"])
    if not abstract:
        continue
    records.append({
        "instruction": f'{TASK}\n\nQuestion: {row["question"].strip()}',
        "input": abstract,
        "output": f'Answer: {row["final_decision"]}\n\n{" ".join(row["long_answer"].split())}',
        "decision": row["final_decision"],
        "pubid": row["pubid"],
    })

print(f"{len(records)} instruction records built")

In [ ]:
# Identical to notebook 02, deliberately — same seed, same ordering, same result. The recipe
# matters: sort the classes, sort each class by pubid, then shuffle. Iterating a dict or relying
# on dataset order would make the "same 200 articles" claim quietly untrue.
rng = random.Random(SEED + 1)
by_decision = {}
for r in records:
    by_decision.setdefault(r["decision"], []).append(r)

train_records, eval_records = [], []
for decision in sorted(by_decision):
    rows = sorted(by_decision[decision], key=lambda r: r["pubid"])
    rng.shuffle(rows)
    n_eval = round(len(rows) * EVAL_FRACTION)
    eval_records.extend(rows[:n_eval])
    train_records.extend(rows[n_eval:])
rng.shuffle(train_records); rng.shuffle(eval_records)

eval_counts = Counter(r["decision"] for r in eval_records)
MAJORITY_LABEL, majority_n = eval_counts.most_common(1)[0]
MAJORITY_BASELINE = majority_n / len(eval_records)

print(f"train {len(train_records)}  {dict(Counter(r['decision'] for r in train_records))}")
print(f"eval  {len(eval_records)}  {dict(eval_counts)}")
print(f"\nmajority-class baseline: {MAJORITY_BASELINE:.1%} (always answer '{MAJORITY_LABEL}')")

## 5. The prompt template, unchanged

Byte-for-byte the template notebook 02 trained on, and the grading regex it was scored with. A
preference stage that quietly reformats its prompts measures the reformatting, not the preference.

In [ ]:
PROMPT_WITH_INPUT = (
    "Below is an instruction describing a task, paired with input providing further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n"
)

PROMPT_NO_INPUT = (
    "Below is an instruction describing a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Response:\n"
)


def build_prompt(record) -> str:
    """The part the model reads but is NOT trained to produce."""
    template = PROMPT_WITH_INPUT if record["input"].strip() else PROMPT_NO_INPUT
    return template.format(instruction=record["instruction"].strip(),
                           input=record["input"].strip())


DECISION_RE = re.compile(r"answer\s*:?\s*\b(yes|no|maybe)\b", re.IGNORECASE)


def parse_decision(text):
    """Notebook 02's grading regex, reused unchanged. None means unparseable."""
    hit = DECISION_RE.search(text)
    return hit.group(1).lower() if hit else None


print(build_prompt(train_records[0])[:420] + " ...")

## 6. Mining preference pairs from the model's own mistakes

A preference pair is three strings: a **prompt**, a **chosen** answer, and a **rejected** answer.
The only interesting question in this notebook is where to get them.

The cheap option is to write a handful by hand, or to manufacture a rejected answer by corrupting
the gold one — flip `yes` to `no`, keep the justification. It runs instantly and it teaches the
model almost nothing, because a corrupted gold answer is trivially separable from a real one. DPO
will happily learn that surface cue and leave the underlying behaviour untouched.

The honest option is **on-policy sampling**. For each training question, sample two answers from
the stage-2 model at a temperature high enough that it sometimes disagrees with itself, grade both
with the same regex notebook 02 was scored with, and build the pair from what comes back:

| samples for one question | chosen | rejected |
|---|---|---|
| one right, one wrong | the model's own correct answer | the model's own wrong answer |
| both wrong | the **expert's** answer | one of the wrong answers |
| both right | *no pair — there is nothing to learn here* | |

The negatives are then genuinely drawn from the distribution being optimised, which is what the DPO
derivation assumes. It costs a full generation pass — the slowest cell in the notebook — so the
result is cached, keyed on the settings that produced it.

In [ ]:
PAIR_PROMPTS = 600            # how many of the 800 training questions to sample from
SAMPLES_PER_PROMPT = 2        # k answers per question; higher k finds more disagreements
SAMPLE_TEMPERATURE = 0.9      # at 0 the model is deterministic and never disagrees with itself
SAMPLE_TOP_P = 0.95
SAMPLE_MAX_NEW_TOKENS = 120   # the same cap notebook 02 scored under
MAX_PROMPT_LEN = 896          # p99 of this dataset is ~714 tokens; nothing truncates

MINING_CONFIG = {
    "model": MODEL_ID, "seed": SEED, "prompts": PAIR_PROMPTS, "k": SAMPLES_PER_PROMPT,
    "temperature": SAMPLE_TEMPERATURE, "top_p": SAMPLE_TOP_P,
    "max_new_tokens": SAMPLE_MAX_NEW_TOKENS,
    "stage1": STAGE1_DIR.name, "stage2": STAGE2_DIR.name,
}

PAIR_CACHE = Path("/content/outputs/stage3-preference-pairs.json")
DRIVE_CACHE = Path("/content/drive/MyDrive/finetuning-demo/stage3-preference-pairs.json")

# A cache is only valid for the settings that produced it. Comparing the whole config dict means
# changing any knob above silently invalidates it, rather than silently reusing the wrong pairs.
pairs = None
for path in (DRIVE_CACHE, PAIR_CACHE):
    if not path.exists():
        continue
    blob = json.loads(path.read_text())
    if blob.get("config") == MINING_CONFIG:
        pairs = blob["pairs"]
        print(f"Loaded {len(pairs)} cached pairs from {path}")
        break
    print(f"{path} was mined with different settings — ignoring it and re-mining.")

if pairs is None:
    print(f"No usable cache. Will sample {PAIR_PROMPTS} x {SAMPLES_PER_PROMPT} = "
          f"{PAIR_PROMPTS * SAMPLES_PER_PROMPT} completions. This is the slow part.")

In [ ]:
@torch.no_grad()
def sample_completions(m, batch, k):
    """k sampled answers per record. Returns one list of k strings per record."""
    m.eval()
    m.config.use_cache = True
    enc = tokenizer([build_prompt(r) for r in batch], return_tensors="pt", padding=True,
                    truncation=True, max_length=MAX_PROMPT_LEN).to("cuda")
    out = m.generate(**enc, max_new_tokens=SAMPLE_MAX_NEW_TOKENS,
                     do_sample=True, temperature=SAMPLE_TEMPERATURE, top_p=SAMPLE_TOP_P,
                     num_return_sequences=k,
                     eos_token_id=tokenizer.eos_token_id,
                     pad_token_id=tokenizer.pad_token_id)
    gen = out[:, enc["input_ids"].shape[1]:]
    texts = [tokenizer.decode(g, skip_special_tokens=True).strip() for g in gen]
    # generate() returns k consecutive rows per input, in input order.
    return [texts[i * k:(i + 1) * k] for i in range(len(batch))]


# One look at what the sampler produces, before committing a quarter of an hour to it.
if pairs is None:
    torch.manual_seed(SEED)
    demo = train_records[0]
    for i, s in enumerate(sample_completions(model, [demo], SAMPLES_PER_PROMPT)[0]):
        print(f"--- sample {i}  (parsed: {parse_decision(s)}, expert: {demo['decision']}) ---")
        print(" ".join(s.split())[:300], "\n")

In [ ]:
# The slow cell: ~12-18 minutes on a T4, skipped entirely on a cache hit.
if pairs is None:
    SAMPLE_BATCH = 4          # x SAMPLES_PER_PROMPT sequences in flight at once
    torch.manual_seed(SEED)

    prompt_pool = train_records[:PAIR_PROMPTS]
    sampled = []
    for i in range(0, len(prompt_pool), SAMPLE_BATCH):
        sampled.extend(sample_completions(model, prompt_pool[i:i + SAMPLE_BATCH],
                                          SAMPLES_PER_PROMPT))
        print(f"\r  sampled {min(i + SAMPLE_BATCH, len(prompt_pool))}/{len(prompt_pool)} questions",
              end="")
    print()

    graded = [[(s, parse_decision(s)) for s in group] for group in sampled]
    flat = [d for group in graded for _, d in group]
    matching = sum(1 for r, group in zip(prompt_pool, graded)
                   for _, d in group if d == r["decision"])
    print(f"\nsampled answers      : {len(flat)}")
    print(f"unparseable          : {sum(1 for d in flat if d is None)}")
    print(f"matching the expert  : {matching / len(flat):.1%}  (at temperature {SAMPLE_TEMPERATURE})")

In [ ]:
if pairs is None:
    pairs = []
    no_wrong = no_right = blank = 0
    for record, group in zip(prompt_pool, graded):
        # An empty generation is neither correct nor incorrect — it is a failed sample, and
        # counting it as either would misreport what the model actually did.
        usable = [(s, d) for s, d in group if s.strip()]
        blank += len(group) - len(usable)

        right = [s for s, d in usable if d == record["decision"]]
        wrong = [s for s, d in usable if d != record["decision"]]

        if not wrong:
            no_wrong += 1
            continue                       # no disagreement, so nothing to learn from
        if not right:
            no_right += 1

        chosen, source = (right[0], "on-policy") if right else (record["output"], "expert")
        rejected = wrong[0]

        # The two sides graded differently, so they cannot be the same string. Guard anyway — a
        # degenerate pair contributes an exactly zero margin, and therefore no gradient.
        if " ".join(chosen.split()) == " ".join(rejected.split()):
            continue

        # No EOS is appended here. TRL adds it to both completions itself for non-conversational
        # datasets; doing it here as well would append it twice. Section 10 checks that it did.
        pairs.append({
            "prompt": build_prompt(record),
            "chosen": chosen,
            "rejected": rejected,
            "chosen_source": source,
            "decision": record["decision"],
            "pubid": record["pubid"],
        })

    print(f"questions sampled            : {len(prompt_pool)}")
    print(f"  no incorrect sample        : {no_wrong}  (skipped — no disagreement to learn from)")
    print(f"  no correct sample          : {no_right}  (chosen falls back to the expert answer)")
    print(f"empty generations discarded  : {blank}")
    print(f"\npreference pairs built       : {len(pairs)}")

assert len(pairs) >= 100, (
    f"Only {len(pairs)} pairs. Raise PAIR_PROMPTS or SAMPLES_PER_PROMPT — DPO on fewer than "
    "~100 pairs measures noise."
)

### Audit the pairs before training on them

Three things quietly ruin a preference dataset, and all three are cheap to check.

**Length bias.** If the rejected answers are systematically longer than the chosen ones, DPO learns
"be brief" and you report it as "learned to be correct". This is the single most common way
preference tuning fools the person running it — length-normalised losses like SimPO exist precisely
because the plain sigmoid loss cannot tell the two apart.

**Provenance.** Pairs whose `chosen` came from the expert rather than the model are *off* policy on
the preferred side. They are still useful, but they push the model toward text it would never have
produced, and they are the usual reason `rewards/chosen` falls during training. Worth knowing the
proportion before reading the curves in section 12.

**Leakage.** No pair may touch the 200 held-out articles, or the final accuracy is measuring a
third round of training on the test set.

In [ ]:
def n_tokens(s):
    return len(tokenizer(s, add_special_tokens=False)["input_ids"])


chosen_len = [n_tokens(p["chosen"]) for p in pairs]
rejected_len = [n_tokens(p["rejected"]) for p in pairs]
gap = sum(rejected_len) / len(pairs) - sum(chosen_len) / len(pairs)

print(f"pairs                 : {len(pairs)}")
print(f"chosen provenance     : {dict(Counter(p['chosen_source'] for p in pairs))}")
print(f"decision mix          : {dict(Counter(p['decision'] for p in pairs))}")
print(f"\nmean tokens, chosen   : {sum(chosen_len) / len(pairs):.1f}")
print(f"mean tokens, rejected : {sum(rejected_len) / len(pairs):.1f}")
print(f"length gap            : {gap:+.1f} tokens")
if abs(gap) > 25:
    print("\n  ^ large enough to matter. Some of any margin gained below is a length preference,")
    print("    not a correctness preference. See the SimPO note in section 16.")
else:
    print("\n  ^ small, so length is not doing the discriminating for us.")

# On-policy pairs only, where both sides genuinely come from the same distribution.
onp = [(c, r) for p, c, r in zip(pairs, chosen_len, rejected_len)
       if p["chosen_source"] == "on-policy"]
if onp:
    print(f"\non-policy pairs only  : chosen {sum(c for c, _ in onp) / len(onp):.1f} tokens, "
          f"rejected {sum(r for _, r in onp) / len(onp):.1f} tokens")

eval_ids = {r["pubid"] for r in eval_records}
overlap = {p["pubid"] for p in pairs} & eval_ids
assert not overlap, f"{len(overlap)} held-out articles leaked into the preference pairs"
print(f"\nheld-out articles in the pair set: {len(overlap)} — the stage-2 test set is still clean.")

In [ ]:
# One complete pair, as DPO will see it.
example = next((p for p in pairs if p["chosen_source"] == "on-policy"), pairs[0])
print("PROMPT (tail):")
print("   ..." + example["prompt"][-300:])
print(f"\nCHOSEN   [{example['chosen_source']}, expert label: {example['decision']}]:")
print("   " + " ".join(example["chosen"].split())[:400])
print(f"\nREJECTED [parsed as: {parse_decision(example['rejected'])}]:")
print("   " + " ".join(example["rejected"].split())[:400])

In [ ]:
# Cache the mined pairs. The generation pass is the expensive part of this notebook and there is
# no reason to pay for it twice.
PAIR_CACHE.parent.mkdir(parents=True, exist_ok=True)
blob = json.dumps({"config": MINING_CONFIG, "pairs": pairs}, ensure_ascii=False)
PAIR_CACHE.write_text(blob, encoding="utf-8")
print(f"wrote {PAIR_CACHE}  ({len(blob) / 1e6:.1f} MB)")

if Path("/content/drive/MyDrive").exists():
    DRIVE_CACHE.parent.mkdir(parents=True, exist_ok=True)
    DRIVE_CACHE.write_text(blob, encoding="utf-8")
    print(f"wrote {DRIVE_CACHE}")

## 7. Hold out some pairs

DPO's own metric — does the model rank the chosen answer above the rejected one? — needs pairs the
model never trained on, for the same reason every other stage needed a held-out split.

These are held out at the **pair** level, from articles that are all inside the stage-2 training
set. They measure whether the learned preference generalises to unseen *pairs*, not whether the
model generalises to unseen *articles*. The 200-article test in section 14 does that.

In [ ]:
from datasets import Dataset

PREF_EVAL_FRACTION = 0.15

rng = random.Random(SEED + 3)
shuffled = sorted(pairs, key=lambda p: p["pubid"])     # sort first, so the shuffle is reproducible
rng.shuffle(shuffled)
n_pref_eval = max(32, round(len(shuffled) * PREF_EVAL_FRACTION))
pref_eval, pref_train = shuffled[:n_pref_eval], shuffled[n_pref_eval:]

KEEP = ["prompt", "chosen", "rejected"]     # the three columns TRL's preference format wants
pref_train_ds = Dataset.from_list([{k: p[k] for k in KEEP} for p in pref_train])
pref_eval_ds = Dataset.from_list([{k: p[k] for k in KEEP} for p in pref_eval])

print(f"train pairs {len(pref_train_ds)}  |  eval pairs {len(pref_eval_ds)}")
print(f"columns: {pref_train_ds.column_names}")
print(f"\narticles shared between the two pair splits: "
      f"{len({p['pubid'] for p in pref_train} & {p['pubid'] for p in pref_eval})}")

## 8. A fresh LoRA for stage 3

Same rank, same targets, same fp32 upcast as the two earlier stages. **Fresh**, not a continuation
of stage 2's adapter — stage 2 is part of the frozen base weights now, and this adapter has a second
job the previous ones did not have: it is what the reference policy gets copied from.

$B$ is zero-initialised, so at step 0 this adapter computes the identity and the policy *is* the
model it started from. That is what makes the reference exact, and it means the DPO reward margin
must start at precisely zero — asserted in section 12, not assumed.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# The same fp16 gotcha as stages 1 and 2: mixed precision keeps an fp32 master copy of the
# trainable parameters, and the gradient scaler refuses to unscale gradients that are themselves
# fp16. Base weights stay in fp16, which is where the memory actually is.
upcast = 0
for name, param in model.named_parameters():
    if param.requires_grad and param.dtype != torch.float32:
        param.data = param.data.float()
        upcast += 1
model.config.use_cache = False   # incompatible with training; generation turns it back on
print(f"\nupcast {upcast} trainable tensors to fp32 (base weights stay in {DTYPE})")

# Flip this to True if section 11 dies with CUDA out of memory. It trades ~30% more compute for a
# large drop in activation memory. With PEFT it also needs enable_input_require_grads(): the frozen
# embedding output would otherwise carry no grad_fn, checkpointing would have nothing to recompute
# through, and you get "element 0 of tensors does not require grad".
USE_GRADIENT_CHECKPOINTING = False
if USE_GRADIENT_CHECKPOINTING:
    model.enable_input_require_grads()
print(f"gradient checkpointing: {USE_GRADIENT_CHECKPOINTING}")

## 9. `DPOConfig` — the defaults that will bite you

`DPOConfig` inherits from `TrainingArguments` through TRL's `BaseConfig`, and **both layers change
defaults on the way**. The changed ones are, with some reliability, exactly the ones that fail
quietly rather than loudly. Every value below is therefore set explicitly.

**Changed by TRL** (`BaseConfig`, then `DPOConfig`):

| setting | TRL's default | here | why |
|---|---|---|---|
| `gradient_checkpointing` | `True` | `False` | 1B fits without it, and with PEFT it needs `enable_input_require_grads()` or training dies on the first backward |
| `bf16` | `None`, resolved to `not fp16` in `__post_init__` | `False` on a T4 | bf16 needs Ampere. Left alone, this switches itself on and a Turing T4 emulates it, slowly |
| `learning_rate` | `1e-6` | `5e-5` | `1e-6` is a full-fine-tune rate; LoRA trains ~0.9% of the weights and needs considerably more |
| `logging_steps` | `10` | `5` | this is a short run — ~80 steps total |
| `loss_type` | `["sigmoid"]` | `["sigmoid"]` | a **list**, not a string, since trl 0.29. Passing `"sigmoid"` is not the same thing |

**Inherited unchanged from `TrainingArguments`**, and still worth setting, because the defaults suit
a different job:

| setting | default | here | why |
|---|---|---|---|
| `per_device_train_batch_size` | `8` | `1` | one DPO *example* is two ~900-token sequences, and the reference forward pass doubles that again |
| `lr_scheduler_type` | `"linear"` | `"cosine"` | matches stages 1 and 2 |
| `save_strategy` | `"steps"`, every 500 | `"no"` | we save the adapter ourselves, in section 13 |

Two settings are specific to DPO and deserve their own paragraph.

**`beta = 0.1`.** How tightly the model is held to the reference. Lower lets it drift further and
chase the preference harder; higher keeps it close and it learns less. As $\beta \to \infty$
nothing moves at all. `0.1` is the value from the paper and a reasonable first guess, not a tuned
one.

**`max_length = 1024`, with `truncation_mode="keep_start"`.** Truncation drops tokens from the
**end** — which is the completion, the only part DPO actually scores. A `max_length` set too low
does not degrade training, it deletes the training signal. Section 10 asserts that nothing
truncates rather than trusting the arithmetic.

In [ ]:
from trl import DPOConfig, DPOTrainer

OUT_DIR = "/content/outputs/stage3-dpo-lora"
BETA = 0.1
DPO_MAX_LENGTH = 1024

dpo_args = DPOConfig(
    output_dir=OUT_DIR,
    beta=BETA,
    loss_type=["sigmoid"],          # the original DPO loss. A list, not a string, in trl 0.29
    max_length=DPO_MAX_LENGTH,
    truncation_mode="keep_start",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,  # effective batch 8 pairs = 16 sequences
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    weight_decay=0.0,
    max_grad_norm=1.0,
    fp16=not SUPPORTS_BF16,
    bf16=SUPPORTS_BF16,
    gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    logging_steps=5,
    eval_strategy="epoch",
    eval_on_start=True,             # captures the step-0 margin, which must be exactly zero
    save_strategy="no",
    report_to="none",
    seed=42,
)

trainer = DPOTrainer(
    model=model,          # already wrapped by get_peft_model, so no peft_config here — passing
    ref_model=None,       # both raises. ref_model=None is what makes the reference free.
    args=dpo_args,
    train_dataset=pref_train_ds,
    eval_dataset=pref_eval_ds,
    processing_class=tokenizer,
)

eff_batch = dpo_args.per_device_train_batch_size * dpo_args.gradient_accumulation_steps
steps = math.ceil(len(pref_train_ds) / eff_batch) * int(dpo_args.num_train_epochs)
print(f"\n{len(pref_train_ds)} pairs, effective batch {eff_batch}, "
      f"{dpo_args.num_train_epochs:g} epochs -> ~{steps} optimizer steps")

### What TRL just did with the reference model

This is the part worth understanding, because "the reference model is free" is repeated far more
often than it is explained, and the usual explanation is wrong.

Given a PEFT model and `ref_model=None`, TRL does **not** simply switch the adapter off whenever it
needs a reference. It adds a **second adapter named `ref`** — a frozen copy of ours, taken at
construction time — and evaluates $\pi_\text{ref}$ through that. Our fresh adapter is
zero-initialised, so the copy is the identity too, and the reference is therefore exactly the
merged stage-2 model from section 3. Exact, and it costs one adapter's worth of memory (~45 MB)
rather than a second 1.24B-parameter model (~2.5 GB). On a 16 GB T4 that is the difference between
fitting and not.

Two consequences follow, and both matter later:

- `trainer.ref_model` is `None`, because there is no separate model object to hold.
- **the model now carries two adapters**, which is why section 13 saves with an explicit
  `selected_adapters=["default"]` — otherwise the artifact ships with a redundant copy of itself.

In [ ]:
# Assert the wiring rather than trusting the docs.
adapters = list(model.peft_config)
print(f"adapters on the model : {adapters}")
print(f"trainer.ref_model     : {type(trainer.ref_model).__name__ if trainer.ref_model is not None else 'None'}")

assert trainer.ref_model is None, (
    "TRL loaded a separate reference model. That is a second full copy of the weights and it will "
    "very likely OOM on a T4 — check that `model` is a PeftModel and `ref_model=None`."
)
assert "ref" in adapters, (
    f"Expected TRL to add a frozen 'ref' adapter, found {adapters}. The reference policy is not "
    "what the section above describes; do not trust the reward numbers until this is understood."
)
ref_trainable = sum(p.numel() for n, p in model.named_parameters() if ".ref." in n and p.requires_grad)
assert ref_trainable == 0, f"{ref_trainable} reference parameters are trainable — the anchor would move"

print("\nreference policy: a frozen copy of the (currently identity) adapter — no second model.")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 10. Check TRL's tokenization before spending ten minutes on it

TRL tokenizes the pairs for you. That is convenient, and it is also the one part of this pipeline
you cannot see, so here is what it produced. Three things matter:

1. **BOS on the prompt**, once, at the front.
2. **EOS at the end of both completions.** Without it the model never learns to stop — the failure
   `docs/concepts.md` §6 covers, and one that a preference stage can *reintroduce* after stage 2
   fixed it, because DPO is free to push probability mass off the EOS token.
3. **Nothing truncated.** `keep_start` removes tokens from the end, and the end is the completion.

The column names TRL uses have changed across releases (`chosen_ids` in trl 0.29, `chosen_input_ids`
in older versions). This cell tries both and **raises** if it recognises neither — a check that
silently skips itself is worse than no check, because it reads as a pass.

In [ ]:
row = trainer.train_dataset[0]
print("TRL produced these columns:", sorted(row.keys()))

NAMES = [("prompt_ids", "chosen_ids", "rejected_ids"),                    # trl >= 0.29
         ("prompt_input_ids", "chosen_input_ids", "rejected_input_ids")]  # older releases
keys = next((k for k in NAMES if k[0] in row and k[1] in row), None)
if keys is None:
    raise RuntimeError(
        f"Unrecognised trl tokenization columns: {sorted(row.keys())}. Do not skip this check — "
        "read the keys above, update NAMES, and re-run. The truncation assertion below is the "
        "only thing standing between you and silently training on cut-off completions."
    )
P, CH, RJ = keys
print(f"reading columns {keys}")

p_ids, c_ids, r_ids = row[P], row[CH], row[RJ]
print(f"\nprompt   {len(p_ids):>4} tokens  starts {p_ids[:3]} (bos={tokenizer.bos_token_id})"
      f"  ...ends {tokenizer.decode(p_ids[-6:])!r}")
print(f"chosen   {len(c_ids):>4} tokens  ends {c_ids[-3:]} (eos={tokenizer.eos_token_id})")
print(f"rejected {len(r_ids):>4} tokens  ends {r_ids[-3:]}")

assert p_ids[0] == tokenizer.bos_token_id, (
    f"Prompt does not start with BOS (got {p_ids[0]}). Stages 1 and 2 both trained with it."
)
assert c_ids[-1] == tokenizer.eos_token_id and r_ids[-1] == tokenizer.eos_token_id, (
    "trl did not append EOS to the completions. This stage would then unlearn how to stop. "
    "Append tokenizer.eos_token to `chosen` and `rejected` in section 6 and re-run from there."
)
print("\nBOS at the front, EOS on both completions.")

# chosen_ids / rejected_ids hold the COMPLETION only — trl slices the prompt back off — so the
# full sequence length is the prompt plus the longer of the two sides.
lengths = [len(x[P]) + max(len(x[CH]), len(x[RJ])) for x in trainer.train_dataset]
over = sum(1 for n in lengths if n > DPO_MAX_LENGTH)
print(f"\nlongest pair: {max(lengths)} tokens vs max_length={DPO_MAX_LENGTH}")
assert over == 0, (
    f"{over} pairs exceed max_length and would have their completions cut off. "
    f"Raise DPO_MAX_LENGTH to at least {max(lengths)}."
)
print(f"pairs truncated: {over}")

## 11. Train

Roughly ten minutes. Each optimizer step runs a policy forward over 16 sequences and a reference
forward over the same 16, so it is about four times the work per example that stage 2 did.

In [ ]:
trainer.train()

# Keep the log before freeing the trainer — the plots below need it, the optimizer state does not.
LOG = list(trainer.state.log_history)

## 12. Reward curves — and the one that actually matters

DPO logs an *implicit reward* for each side of every pair:

$$r(y) = \beta \log \frac{\pi_\theta(y \mid x)}{\pi_\text{ref}(y \mid x)}$$

It is not a reward model's opinion about quality. It is only how much more, or less, likely the
tuned model finds an answer than the model it started from. Three numbers come out of it:

- **`rewards/margins`** — chosen minus rejected. This is what DPO maximises, and it should rise.
- **`rewards/accuracies`** — how often that margin is positive. The objective's own accuracy.
- **`rewards/chosen`** — and this is the one to actually look at.

Here is the part most write-ups leave out. **`rewards/chosen` usually goes negative.** The margin
can be widened either by raising the chosen log-probability or by lowering the rejected one, and
gradient descent overwhelmingly picks the second, because making text less likely is far easier
than making it more likely. So a rising margin is entirely compatible with the model becoming
*less* confident about good answers — just more slowly than it becomes less confident about bad
ones.

If `rewards/chosen` falls off a cliff, the model is not learning a preference, it is suppressing
its own output distribution. Lower the learning rate or raise $\beta$.

In [ ]:
train_log = [e for e in LOG if "loss" in e and "eval_loss" not in e]
eval_log = [e for e in LOG if "eval_loss" in e]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

axes[0].plot([e["step"] for e in train_log], [e["loss"] for e in train_log],
             color="#4C72B0", label="train")
if eval_log:
    axes[0].plot([e["step"] for e in eval_log], [e["eval_loss"] for e in eval_log],
                 color="#C44E52", marker="o", label="eval")
axes[0].axhline(math.log(2), color="#888888", ls=":", lw=1, label="log 2 = no preference")
axes[0].set_title("DPO loss"); axes[0].set_xlabel("step"); axes[0].legend()
axes[0].grid(alpha=0.3)

for key, colour, label in [("rewards/chosen", "#55A868", "chosen"),
                           ("rewards/rejected", "#C44E52", "rejected"),
                           ("rewards/margins", "#4C72B0", "margin")]:
    pts = [(e["step"], e[key]) for e in train_log if key in e]
    if pts:
        axes[1].plot([s for s, _ in pts], [v for _, v in pts], color=colour, label=label)
axes[1].axhline(0, color="#888888", lw=1)
axes[1].set_title("implicit rewards"); axes[1].set_xlabel("step"); axes[1].legend()
axes[1].grid(alpha=0.3)

for log, colour, label, key in [(train_log, "#4C72B0", "train", "rewards/accuracies"),
                                (eval_log, "#C44E52", "eval", "eval_rewards/accuracies")]:
    pts = [(e["step"], e[key]) for e in log if key in e]
    if pts:
        axes[2].plot([s for s, _ in pts], [v for _, v in pts], color=colour,
                     marker="o" if label == "eval" else None, label=label)
axes[2].axhline(0.5, color="#888888", ls=":", lw=1, label="coin flip")
axes[2].set_ylim(0, 1.02)
axes[2].set_title("preference accuracy"); axes[2].set_xlabel("step"); axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# At step 0 the LoRA B matrices are still zero, so the policy IS the reference and every implicit
# reward must be exactly zero. eval_on_start=True captured that moment. If the margin is not ~0
# there, the reference policy is not the model you think it is, and nothing below means anything.
if eval_log:
    first, last = eval_log[0], eval_log[-1]
    print(f"{'held-out pairs':22s}{'step 0':>12s}{'final':>12s}")
    print("-" * 46)
    for key, name in [("eval_rewards/margins", "reward margin"),
                      ("eval_rewards/chosen", "reward, chosen"),
                      ("eval_rewards/rejected", "reward, rejected"),
                      ("eval_rewards/accuracies", "preference acc."),
                      ("eval_loss", "loss")]:
        if key in first and key in last:
            print(f"{name:22s}{first[key]:>12.4f}{last[key]:>12.4f}")
    print("-" * 46)

    start_margin = first.get("eval_rewards/margins")
    if start_margin is not None:
        assert abs(start_margin) < 1e-2, (
            f"Step-0 margin is {start_margin:.4f}, not ~0. The reference policy differs from the "
            "initial policy — check that no adapter was already active before training."
        )
        print(f"\nstep-0 margin {start_margin:+.2e} — policy and reference were identical, as they must be.")
        print("(preference accuracy reads 0.0 there, not 0.5: an exact tie is not scored as a win.)")

## 13. Save the stage-3 adapter

Saved **before** the evaluation below, so that a Colab timeout during a ten-minute generation pass
costs you the numbers rather than the training.

One argument here appears nowhere else in this repo: `selected_adapters=["default"]`. The model has
carried two adapters since section 9 — ours and TRL's frozen `ref` copy — and the default behaviour
would write both. The `ref` copy is an artifact of how the reference policy is implemented, not
something anyone should load later.

In [ ]:
import shutil

ADAPTER_DIR = Path(OUT_DIR)
model.save_pretrained(ADAPTER_DIR, selected_adapters=["default"])
# Saved with padding_side="left" — correct for generation, and deliberately different from the
# stage-1 and stage-2 tokenizers, which were saved for training.
tokenizer.save_pretrained(ADAPTER_DIR)

size = sum(f.stat().st_size for f in ADAPTER_DIR.rglob("*") if f.is_file())
print(f"{ADAPTER_DIR}  ({size / 1e6:.1f} MB)")
for f in sorted(ADAPTER_DIR.iterdir()):
    print(f"  {f.name}{'/' if f.is_dir() else ''}")
assert not (ADAPTER_DIR / "ref").exists(), "the frozen reference adapter was saved too"

if Path("/content/drive/MyDrive").exists():
    dest = Path("/content/drive/MyDrive/finetuning-demo/stage3-dpo-lora")
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(ADAPTER_DIR, dest)
    print(f"\nCopied to {dest}")
else:
    print("\nNot running in Colab — the adapter is on local disk only.")

In [ ]:
# The optimizer state and TRL's cached batches are done with, and generation wants the room.
del trainer
gc.collect(); torch.cuda.empty_cache()
print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 14. The metric that carries over: decision accuracy

A rising reward margin says DPO optimised what DPO optimises. That is necessary, and on its own it
is not interesting — a model can win the preference objective by learning some incidental property
of how the pairs were built.

So we go back to notebook 02's test: **the same 200 held-out articles, the same greedy decoding,
the same grading regex**. Two systems, scored back to back on identical inputs:

| system | how |
|---|---|
| `+ domain + instruct` | `model.disable_adapter()` — the stage-2 policy, exactly |
| `+ domain + instruct + dpo` | adapter on |

`base` and `+ domain` are not re-run here. Notebook 02 already scored them and neither has
changed.

In [ ]:
@torch.no_grad()
def generate_batch(m, batch, max_new_tokens=120):
    """Greedy, left-padded batched generation. Deterministic, so the two systems are comparable."""
    m.eval()
    m.config.use_cache = True
    enc = tokenizer([build_prompt(r) for r in batch], return_tensors="pt", padding=True,
                    truncation=True, max_length=MAX_PROMPT_LEN).to("cuda")
    out = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                     eos_token_id=tokenizer.eos_token_id,
                     pad_token_id=tokenizer.pad_token_id)
    gen = out[:, enc["input_ids"].shape[1]:]
    return [tokenizer.decode(g, skip_special_tokens=True).strip() for g in gen]


def score(m, dataset, label, batch_size=8):
    """Decision accuracy + parse rate over the held-out set."""
    texts = []
    for i in range(0, len(dataset), batch_size):
        texts.extend(generate_batch(m, dataset[i:i + batch_size]))
        print(f"\r  {label}: {min(i + batch_size, len(dataset))}/{len(dataset)}", end="")
    print()
    preds = [parse_decision(t) for t in texts]
    correct = sum(1 for p, r in zip(preds, dataset) if p == r["decision"])
    parsed = sum(1 for p in preds if p is not None)
    return {"accuracy": correct / len(dataset), "parse_rate": parsed / len(dataset),
            "preds": preds, "texts": texts}

In [ ]:
# Two full passes over 200 records. A few minutes.
results = {}
results["+ domain + instruct + dpo"] = score(model, eval_records, "stage 3")
# disable_adapter() switches off every adapter on the model, including TRL's frozen 'ref' copy,
# leaving the merged stage-2 weights from section 3 — the exact model stage 3 started from.
with model.disable_adapter():
    results["+ domain + instruct"] = score(model, eval_records, "stage 2")
print("\ndone")

### Read the difference before you believe it

Two accuracies on 200 records look precise, and are not. The standard error of a proportion near
55% at $n=200$ is about **3.5 points**, so a naive 95% interval around a single number spans
roughly ±7 points. A good deal of published fine-tuning improvement is smaller than that.

The comparison here is better than a naive one, because both systems answered the *same* 200
questions — a paired design. **McNemar's test** uses only the records where the two systems
disagree, which is the only place information about the difference lives, and asks whether those
disagreements split more lopsidedly than a coin flip would explain.

If the p-value is large, the correct conclusion is "no detectable change", and reporting the
accuracy delta as an improvement would be wrong regardless of which way it points.

In [ ]:
gold = [r["decision"] for r in eval_records]
p2 = results["+ domain + instruct"]["preds"]
p3 = results["+ domain + instruct + dpo"]["preds"]

print(f"Held-out set: {len(eval_records)} records   {dict(Counter(gold))}\n")
print(f"{'system':30s}{'accuracy':>10s}{'parse rate':>13s}")
print("-" * 53)
print(f"{'majority class (' + MAJORITY_LABEL + ')':30s}{MAJORITY_BASELINE:>9.1%}{'n/a':>13s}")
for name in ["+ domain + instruct", "+ domain + instruct + dpo"]:
    r = results[name]
    print(f"{name:30s}{r['accuracy']:>9.1%}{r['parse_rate']:>12.1%}")
print("-" * 53)

delta = results["+ domain + instruct + dpo"]["accuracy"] - results["+ domain + instruct"]["accuracy"]
print(f"\nstage 3 vs stage 2 : {delta:+.1%}")
print(f"stage 3 vs majority: {results['+ domain + instruct + dpo']['accuracy'] - MAJORITY_BASELINE:+.1%}")

# McNemar, exact binomial on the discordant pairs only. No scipy: it is three lines.
b = sum(1 for a, c, g in zip(p2, p3, gold) if a == g and c != g)   # stage 2 right, stage 3 wrong
c_ = sum(1 for a, c, g in zip(p2, p3, gold) if a != g and c == g)  # stage 2 wrong, stage 3 right
n = b + c_
p_value = 1.0
if n:
    lo = min(b, c_)
    p_value = min(1.0, 2 * sum(math.comb(n, k) for k in range(lo + 1)) / 2 ** n)

print(f"\nMcNemar, paired on the same {len(gold)} records")
print(f"  agree                        : {len(gold) - n}")
print(f"  stage 2 right, stage 3 wrong : {b}")
print(f"  stage 2 wrong, stage 3 right : {c_}")
print(f"  two-sided exact p            : {p_value:.3f}")
print("\n  " + ("A change this size is not distinguishable from noise at n=200."
                if p_value >= 0.05 else
                "The disagreements are lopsided enough to be unlikely under no true difference."))

### Did preference tuning break anything?

Three regressions are specific to DPO, and all three are invisible in an accuracy number.

**Length drift.** The best-known DPO failure. If the mined rejected answers skewed long, the model
learned brevity; if they skewed short, verbosity. Either way it is not what we meant to teach, and
the length audit in section 6 predicted it.

**Stopping.** Stage 2 taught the model to emit EOS. DPO optimises a ratio, and can push probability
mass off the EOS token as a side effect, so answers start running to the token cap again.

**Label prior.** The cheapest way to move accuracy on a 55/34/11 split is to answer "yes" more
often. Only the confusion matrix shows whether that is what happened.

In [ ]:
def gen_len(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


print(f"{'system':30s}{'mean tokens':>13s}{'hit the 120 cap':>18s}{'parse rate':>13s}")
print("-" * 74)
for name in ["+ domain + instruct", "+ domain + instruct + dpo"]:
    lens = [gen_len(t) for t in results[name]["texts"]]
    capped = sum(1 for n_ in lens if n_ >= 118)
    print(f"{name:30s}{sum(lens) / len(lens):>13.1f}{capped:>15d}/{len(lens)}"
          f"{results[name]['parse_rate']:>12.1%}")
print("-" * 74)
print("\nA rising mean, or more answers hitting the cap, means the model is unlearning EOS.")

In [ ]:
LABELS = ["yes", "no", "maybe"]
for name in ["+ domain + instruct", "+ domain + instruct + dpo"]:
    preds = results[name]["preds"]
    print(f"\n{name}   (rows = expert label, cols = prediction)")
    print(f"{'':8s}" + "".join(f"{c:>8s}" for c in LABELS) + f"{'unparsed':>10s}")
    for actual in LABELS:
        row_counts = [sum(1 for p, g in zip(preds, gold) if g == actual and p == col)
                      for col in LABELS]
        unparsed = sum(1 for p, g in zip(preds, gold) if g == actual and p is None)
        print(f"{actual:8s}" + "".join(f"{v:>8d}" for v in row_counts) + f"{unparsed:>10d}")
    print(f"  predictions: {dict(Counter(p or 'unparsed' for p in preds))}")

print(f"\nexpert distribution: {dict(Counter(gold))}")
print("If stage 3 simply predicts the majority class more often, accuracy can rise while the model")
print("gets less useful. That is visible here and nowhere else.")

## 15. Qualitative side-by-side

The same four held-out articles, before and after preference tuning, against the expert answer.
The numbers tell you whether something moved; this tells you what moved.

In [ ]:
def show(text, limit=420):
    text = " ".join(text.split())
    return text if len(text) <= limit else text[:limit] + " ..."


for i in range(4):
    r = eval_records[i]
    print("=" * 112)
    print(f"Q: {r['instruction'].split('Question: ')[-1]}")
    print(f"   [abstract] {show(r['input'], 180)}\n")
    print(f"  [2] + instruct       : {show(results['+ domain + instruct']['texts'][i])}\n")
    print(f"  [3] + instruct + dpo : {show(results['+ domain + instruct + dpo']['texts'][i])}\n")
    print(f"  EXPERT               : {show(r['output'])}")
print("=" * 112)

## 16. What you built

Three stages, one dataset, one free T4:

```
unsloth/Llama-3.2-1B  (base, no instruction tuning)
  └─ stage 1: LoRA on ~292k tokens of raw abstracts        → writes like a biomedical paper
       └─ merged into the base weights
            └─ stage 2: LoRA on 800 expert (question, answer) pairs
                 │                                          → answers in the right format, and stops
                 └─ merged into the base weights
                      └─ stage 3: LoRA on preference pairs it generated itself
                                                            → ranks its own good answers above its bad ones
```

The third stage is different in kind from the first two. There was no gold target to imitate, only
a comparison — and the training signal came from the model's own outputs, graded against a label it
had already been trained on. That is the same shape as RLHF, with the reward model and the RL loop
collapsed into a single classification loss.

### The three things this stage is built to keep you honest about

1. **The reference model is not free unless you make it free, and it is not what people say it is.**
   `ref_model=None` on a PEFT model makes TRL clone the adapter into a frozen `ref` copy — not a
   second 2.5 GB model, and not a bare `disable_adapter()` either. The step-0 margin assertion
   proves the reference is the model we think it is, rather than assuming it.
2. **A rising margin is not a better model.** `rewards/chosen` typically falls, because the margin
   is cheaper to widen by suppression than by promotion. The accuracy test in section 14 is the
   only thing that can tell those two apart.
3. **A 200-record accuracy delta under about 7 points is noise.** The McNemar test is here because
   reporting the delta alone would have been the easier and worse choice.

### Honest limitations

- **These are proxy preferences, not human ones.** "Chosen" means "agreed with the expert's
  yes/no/maybe". Real preference data encodes helpfulness, tone, hedging, refusal — none of which a
  regex can grade. This pipeline teaches correctness-against-a-label and nothing beyond it.
- **The preferred side is partly off-policy.** Pairs where every sample was wrong fall back to the
  expert's answer, which the model would never have written. That is the usual cause of a falling
  `rewards/chosen`, and the provenance mix in section 6 says how much of the data it affects.
- **One $\beta$, one learning rate, no sweep.** Both were picked as reasonable defaults. DPO is
  considerably more sensitive to both than instruction tuning is.
- **A few hundred pairs is very few.** Published DPO runs use tens of thousands.
- **The pairs come from articles the model was instruction-tuned on**, so the preference signal
  concerns answers to familiar questions. The 200-article test is still clean; the pair set is not
  a measure of generalisation.
- **Nothing here is medical advice.** A 1B model fine-tuned on 800 abstracts is a training-mechanics
  exercise, not a clinical tool.

### Where to go next

| Change | Why |
|---|---|
| **More pairs** — raise `PAIR_PROMPTS`, or `SAMPLES_PER_PROMPT` to 4 | The highest return here by a distance. k=4 finds far more disagreements per question. |
| **`loss_type=["sigmoid_norm"]` (SimPO)** | Length-normalises the loss. The direct fix if the length audit in section 6 showed a real gap. |
| **ORPO** | Folds preference into the SFT loss and needs *no* reference model at all — one stage instead of two. |
| **KTO** | Takes thumbs-up / thumbs-down on single answers rather than pairs. Much cheaper to collect. |
| **A $\beta$ sweep** — 0.01, 0.1, 0.5 | The one hyperparameter that changes *what* DPO does, rather than how fast it does it. |
| **An LLM judge instead of the regex** | Grades helpfulness and grounding, not just the label. The step toward preferences worth the name. |
| **`precompute_ref_log_probs=True`** | The reference log-probs are constant; computing them once trades memory for a shorter epoch. |
| **RLHF proper — reward model + PPO/GRPO** | What DPO is a shortcut for. Worth doing once, to see what the shortcut removed. |

See `docs/concepts.md` §10 and §11 for the reasoning behind the choices in this notebook.